# Experiment: Fixed Scan-Budget Grid for MCMC-IS

Objective:
- At a fixed total MCMC-IS budget of 10 million evaluations, compare several fixed beta-selection budgets.
- For each scan budget, choose beta by the same local scan objective and then spend the remaining budget on production.
- Avoid duplicate production work by grouping identical selected beta/proposal pairs and evaluating all required production cutoffs from one cumulative run.

This is an offline calibration tool. After the run, we can fit a coarse rule of thumb such as scan budget as a function of total budget and/or \(p_0\).

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_cross_method_saved_output,
    load_beta_sweep_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    plot_named_method_convergence,
    plot_named_method_max_budget,
    read_json,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
    tune_samc_setup,
    write_json,
    write_jsonl,
    _effective_n_jobs,
    _iid_replicate_worker,
    _samc_replicate_worker,
    _try_make_process_pool,
)

pd.set_option("display.max_columns", 100)
project_root

In [ ]:
import concurrent.futures as cf
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from perm_pval.core.proposals import resolve_n_swap_pairs
from perm_pval.diagnostics.is_weights import effective_sample_size, summarize_weights
from perm_pval.experiments.notebook_studies import (
    _annotate_error_fields,
    _burn_in,
    _effective_n_jobs,
    _kept_samples_per_chain,
    _mcmc_eval_count,
    _mcmc_prefix_row,
    _run_single_chain_full_trace,
    _steps_per_chain,
    _try_make_process_pool,
    local_beta_scan,
    run_scan_budget_repeat_job,
)
from perm_pval.methods.beta_tuning import estimate_scale_T, iid_pilot_statistics

## Configuration

`TOTAL_BUDGET` is the full MCMC-IS evaluation budget: IID pilot + beta scan + production. `SCAN_BUDGETS` below are target beta-selection budgets, including the IID pilot. The actual charged budget can differ by a few evaluations because MCMC chains are split across chains and candidates.

The scan-budget grid is deliberately simple: it asks whether spending roughly `50k`, `100k`, `500k`, `1m`, or `2m` on beta selection is worthwhile when the full budget is `10m`.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_scan_budget_grid"

SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_sig_n60",
    "linear_stat_dp_n40",
    "rank_sum_dp_n40",
]
MIN_TAIL_STATES = 2

TOTAL_BUDGET = 10_000_000 if not FAST_MODE else 200_000
N_REPEATS = 6 if not FAST_MODE else 1
N_JOBS = min(6, os.cpu_count() or 1) if not FAST_MODE else 1
BASE_SEED = 83_083

SCAN_BUDGETS = (
    50_000,
    100_000,
    500_000,
    1_000_000,
    2_000_000,
) if not FAST_MODE else (25_000, 50_000)
SCAN_FINALIST_COUNT = 4
SCAN_REFINE_TO_SCREEN_RATIO = 1.0
SCAN_FINAL_TO_SCREEN_RATIO = 2.0

Q_MULTIPLIERS = (0.001, 0.005, 0.01, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.33, 0.5, 1.0)
COARSE_Q_MULTIPLIERS = (0.001, 0.01, 0.10, 0.5)
MCMC_LOCAL_SCAN_STRATEGY = "adaptive_q"
MCMC_LOCAL_SCAN_OBJECTIVE = "varhat_qmatch_soft"
PRODUCTION_ESTIMATE_VARIANCE = False
MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND = {
    "small": 1,
    "medium": 1,
    "large": 2,
}

BUDGET_RULE_SPECS = [
    {
        "name": f"scan_{budget // 1_000_000}m" if budget >= 1_000_000 else f"scan_{budget // 1_000}k",
        "kind": "fixed_scan_budget",
        "target_beta_selection_budget": int(budget),
        "finalist_count": SCAN_FINALIST_COUNT,
        "refine_to_screen_ratio": SCAN_REFINE_TO_SCREEN_RATIO,
        "final_to_screen_ratio": SCAN_FINAL_TO_SCREEN_RATIO,
    }
    for budget in SCAN_BUDGETS
]

base_mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=25_000 if not FAST_MODE else 5_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=1,
    local_scan_strategy=MCMC_LOCAL_SCAN_STRATEGY,
    local_scan_objective=MCMC_LOCAL_SCAN_OBJECTIVE,
    local_scan_q_multipliers=Q_MULTIPLIERS,
    local_scan_coarse_q_multipliers=COARSE_Q_MULTIPLIERS,
    local_scan_screen_chains=1,
    local_scan_refine_top_k=2,
    local_scan_refine_radius=1,
    local_scan_refine_max_q_points=6,
    local_scan_chains=2,
    local_scan_burn_in_fraction=0.20,
    local_scan_thin=1,
)

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "TOTAL_BUDGET": TOTAL_BUDGET,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SCAN_BUDGETS": SCAN_BUDGETS,
    "SCAN_FINALIST_COUNT": SCAN_FINALIST_COUNT,
    "SCAN_REFINE_TO_SCREEN_RATIO": SCAN_REFINE_TO_SCREEN_RATIO,
    "SCAN_FINAL_TO_SCREEN_RATIO": SCAN_FINAL_TO_SCREEN_RATIO,
    "Q_MULTIPLIERS": Q_MULTIPLIERS,
    "COARSE_Q_MULTIPLIERS": COARSE_Q_MULTIPLIERS,
    "MCMC_LOCAL_SCAN_STRATEGY": MCMC_LOCAL_SCAN_STRATEGY,
    "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
    "PRODUCTION_ESTIMATE_VARIANCE": PRODUCTION_ESTIMATE_VARIANCE,
    "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND,
    "BUDGET_RULE_SPECS": BUDGET_RULE_SPECS,
}, indent=2))

## Helpers

In [ ]:
def mcmc_proposal_size_for_scenario(scenario) -> int:
    band = str(scenario.portfolio.get("sample_size_band", "medium"))
    return int(MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND.get(band, 1))


def q_target_from_p0(p0: float, cfg: MCMCWorkflowConfig) -> float:
    return float(float(p0) ** float(cfg.d_alpha))


def kept_samples_total(total_steps: int, *, n_chains: int, burn_in_fraction: float, thin: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    burn_in = _burn_in(steps_per_chain, float(burn_in_fraction))
    return int(n_chains) * _kept_samples_per_chain(steps_per_chain, burn_in, int(thin))


def stage_eval_total(total_steps: int, *, n_candidates: int, n_chains: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    return int(n_candidates) * _mcmc_eval_count(steps_per_chain, int(n_chains))


def split_fixed_scan_budget(
    *,
    target_beta_selection_budget: int,
    cfg: MCMCWorkflowConfig,
    n_candidates: int,
    finalist_count: int,
    final_to_screen_ratio: float,
) -> tuple[int, int]:
    scan_budget_ex_pilot = max(int(target_beta_selection_budget) - int(cfg.pilot_samples), 1)
    budget_units = float(n_candidates) + float(finalist_count) * float(final_to_screen_ratio)
    screen_total_steps = max(int(scan_budget_ex_pilot / max(budget_units, 1.0)), 1)
    final_total_steps = max(int(round(float(final_to_screen_ratio) * screen_total_steps)), 1)
    return int(screen_total_steps), int(final_total_steps)


def resolve_budget_rule(rule: dict, scenario, cfg: MCMCWorkflowConfig) -> dict:
    proposal_size = int(mcmc_proposal_size_for_scenario(scenario))
    n_candidates = len(tuple(cfg.local_scan_q_multipliers))
    finalist_count = min(int(rule.get("finalist_count", cfg.local_scan_finalist_count)), n_candidates)
    q_target = q_target_from_p0(scenario.exact_p, cfg)

    if rule["kind"] != "fixed_scan_budget":
        raise ValueError(f"Unknown budget rule kind for this notebook: {rule['kind']}")

    target_beta_selection_budget = int(rule["target_beta_selection_budget"])
    if target_beta_selection_budget >= int(TOTAL_BUDGET):
        raise ValueError("target_beta_selection_budget must be smaller than TOTAL_BUDGET")
    final_to_screen_ratio = float(rule.get("final_to_screen_ratio", 2.0))
    screen_total_steps, final_total_steps = split_fixed_scan_budget(
        target_beta_selection_budget=target_beta_selection_budget,
        cfg=cfg,
        n_candidates=n_candidates,
        finalist_count=finalist_count,
        final_to_screen_ratio=final_to_screen_ratio,
    )

    screen_eval_total = stage_eval_total(
        screen_total_steps,
        n_candidates=n_candidates,
        n_chains=int(cfg.local_scan_screen_chains),
    )
    final_eval_total_upper = stage_eval_total(
        final_total_steps,
        n_candidates=finalist_count,
        n_chains=int(cfg.local_scan_chains),
    )
    expected_beta_selection_budget = int(cfg.pilot_samples) + int(screen_eval_total) + int(final_eval_total_upper)
    return {
        "rule_name": str(rule["name"]),
        "rule_kind": str(rule["kind"]),
        "proposal_size": proposal_size,
        "n_candidates": int(n_candidates),
        "finalist_count": int(finalist_count),
        "q_target": float(q_target),
        "q_ref": np.nan,
        "tail_hit_steps": np.nan,
        "screen_cap_steps": np.nan,
        "target_beta_selection_budget": int(target_beta_selection_budget),
        "target_scan_budget_ex_pilot": int(max(target_beta_selection_budget - int(cfg.pilot_samples), 1)),
        "final_to_screen_ratio": float(final_to_screen_ratio),
        "screen_total_steps": int(screen_total_steps),
        "final_total_steps": int(final_total_steps),
        "expected_screen_eval_total": int(screen_eval_total),
        "expected_final_eval_total_upper": int(final_eval_total_upper),
        "expected_beta_selection_budget_upper": int(expected_beta_selection_budget),
    }


def mcmc_cfg_for_budget_rule(rule_budget: dict, cfg: MCMCWorkflowConfig) -> MCMCWorkflowConfig:
    return replace(
        cfg,
        proposal_size=int(rule_budget["proposal_size"]),
        local_scan_swap_counts=(int(rule_budget["proposal_size"]),),
        local_scan_finalist_count=int(rule_budget["finalist_count"]),
        local_scan_screen_total_steps=int(rule_budget["screen_total_steps"]),
        local_scan_total_steps=int(rule_budget["final_total_steps"]),
    )


def selected_scan_row(scan: dict) -> dict:
    rows = list(scan.get("rows", []))
    if not rows:
        return {}
    beta = float(scan.get("selected_beta", np.nan))
    prop = int(scan.get("selected_proposal_size", -1))
    matches = [
        row
        for row in rows
        if int(row.get("n_swap_pairs", -999)) == prop
        and np.isfinite(row.get("beta", np.nan))
        and abs(float(row.get("beta")) - beta) <= 1e-10
    ]
    return dict(matches[0] if matches else rows[0])

## Load Scenarios

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None,
    min_tail_states=MIN_TAIL_STATES,
)
scenario_by_key = {scenario.key: scenario for scenario in scenarios}
run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "scan_budget_grid") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "family": s.portfolio.get("family"),
        "statistic_family": s.portfolio.get("statistic_family"),
        "sample_size_band": s.portfolio.get("sample_size_band"),
        "proposal_size": mcmc_proposal_size_for_scenario(s),
    }
    for s in scenarios
])

## Run Fixed-Budget Scans

Each `(scenario, repeat)` job uses one common IID pilot and reuses it across all scan budgets. Jobs are independent, so the notebook can parallelize at that outer level while still charging the pilot cost to every budget setting.

In [ ]:
scan_records: list[dict] = []
production_records: list[dict] = []
repeat_jobs: list[dict] = []

for scenario_idx, scenario in enumerate(scenarios):
    rule_budgets = [resolve_budget_rule(rule, scenario, base_mcmc_cfg) for rule in BUDGET_RULE_SPECS]
    print(
        f"Prepared scenario {scenario.key} | exact p={scenario.exact_p:.3e} "
        f"| repeats={N_REPEATS} | rules={len(rule_budgets)}"
    )
    for repeat_idx in range(N_REPEATS):
        pilot_seed = BASE_SEED + 100_000 * scenario_idx + 10_000 * repeat_idx
        repeat_jobs.append(
            {
                "scenario_key": str(scenario.key),
                "scenario_display": str(scenario.description),
                "family": scenario.portfolio.get("family"),
                "statistic_family": scenario.portfolio.get("statistic_family"),
                "sample_size_band": scenario.portfolio.get("sample_size_band"),
                "problem": scenario.problem,
                "exact_p": float(scenario.exact_p),
                "repeat_idx": int(repeat_idx),
                "pilot_seed": int(pilot_seed),
                "total_budget": int(TOTAL_BUDGET),
                "base_mcmc_cfg": base_mcmc_cfg,
                "rule_budgets": [dict(rule_budget) for rule_budget in rule_budgets],
                "production_estimate_variance": bool(PRODUCTION_ESTIMATE_VARIANCE),
            }
        )

t0 = time.perf_counter()
n_workers = _effective_n_jobs(N_JOBS, len(repeat_jobs))
executor = _try_make_process_pool(n_workers) if n_workers > 1 else None
print(f"Running {len(repeat_jobs)} scenario-repeat jobs with n_workers={n_workers}")

def _consume_job_result(result: dict) -> None:
    scan_records.extend(result["scan_records"])
    production_records.extend(result["production_records"])


if executor is None:
    for job_idx, job in enumerate(repeat_jobs, start=1):
        _consume_job_result(run_scan_budget_repeat_job(**job))
        if job_idx % max(1, len(repeat_jobs) // 6) == 0 or job_idx == len(repeat_jobs):
            print(f"Completed {job_idx}/{len(repeat_jobs)} jobs")
else:
    with executor:
        futures = [executor.submit(run_scan_budget_repeat_job, **job) for job in repeat_jobs]
        for job_idx, future in enumerate(cf.as_completed(futures), start=1):
            _consume_job_result(future.result())
            if job_idx % max(1, len(repeat_jobs) // 6) == 0 or job_idx == len(repeat_jobs):
                print(f"Completed {job_idx}/{len(repeat_jobs)} jobs")

scan_df = pd.DataFrame(scan_records)
production_df = pd.DataFrame(production_records)
print(
    f"Completed {len(scan_df)} scans and {len(production_df)} production rows "
    f"in {time.perf_counter() - t0:.1f}s"
)
display(scan_df.sort_values(["scenario", "repeat", "rule_name"])[[
    "scenario", "repeat", "rule_name", "beta_selection_budget", "production_budget",
    "screen_total_steps", "refine_total_steps", "final_total_steps",
    "scan_n_weighted_samples", "selected_scan_n_weighted_samples", "selected_q_multiplier",
    "selected_beta", "selected_q_tilt_tail_share", "selected_ess", "selected_weight_cv",
]])

## Run Deduplicated Production

Production is already executed inside each parallel `(scenario, repeat)` job. Within a job, if multiple scan budgets select the same beta/proposal, we run one cumulative production chain up to the largest required leftover budget and read off the shorter checkpoints for the other scan budgets.

We intentionally do not reuse scan final states here. That keeps the production runs groupable and isolates the beta/proposal selection effect from differences in scan-generated initialization states.

In [ ]:
print(f"Loaded {len(production_df)} policy production rows from the completed scenario-repeat jobs")
display(production_df.sort_values(["scenario", "repeat", "rule_name", "estimator_variant"])[[
    "scenario", "repeat", "rule_name", "estimator_variant", "estimate", "exact_p", "root_squared_error",
    "abs_log10_error", "selected_q_multiplier", "selected_beta", "beta_selection_budget",
    "production_budget", "available_scan_n_weighted_samples", "scan_n_weighted_samples",
    "production_n_weighted_samples", "ess", "weight_cv", "acceptance_rate",
]])

## Summarize Fixed Scan Budgets

In [ ]:
def summarize_policy_records(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (scenario, rule_name, estimator_variant), sub in df.groupby(["scenario", "rule_name", "estimator_variant"], sort=True):
        exact_p = float(sub["exact_p"].iloc[0])
        estimates = sub["estimate"].astype(float).to_numpy()
        rows.append({
            "scenario": scenario,
            "rule_name": rule_name,
            "estimator_variant": estimator_variant,
            "n_runs": int(len(sub)),
            "exact_p": exact_p,
            "mean_estimate": float(np.mean(estimates)),
            "rmse": float(np.sqrt(np.mean((estimates - exact_p) ** 2))),
            "mean_abs_log10_error": float(np.nanmean(sub["abs_log10_error"].astype(float))),
            "mean_selected_q_multiplier": float(np.nanmean(sub["selected_q_multiplier"].astype(float))),
            "median_selected_q_multiplier": float(np.nanmedian(sub["selected_q_multiplier"].astype(float))),
            "mean_beta_selection_budget": float(np.mean(sub["beta_selection_budget"].astype(float))),
            "mean_production_budget": float(np.mean(sub["production_budget"].astype(float))),
            "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
            "mean_ess": float(np.nanmean(sub["ess"].astype(float))),
            "mean_weight_cv": float(np.nanmean(sub["weight_cv"].astype(float))),
            "mean_acceptance_rate": float(np.nanmean(sub["acceptance_rate"].astype(float))),
            "mean_available_scan_n_weighted_samples": float(np.nanmean(sub["available_scan_n_weighted_samples"].astype(float))),
            "mean_available_selected_scan_n_weighted_samples": float(np.nanmean(sub.get("available_selected_scan_n_weighted_samples", pd.Series(dtype=float)).astype(float))) if "available_selected_scan_n_weighted_samples" in sub else np.nan,
            "mean_scan_n_weighted_samples": float(np.nanmean(sub["scan_n_weighted_samples"].astype(float))),
            "mean_production_n_weighted_samples": float(np.nanmean(sub["production_n_weighted_samples"].astype(float))),
        })
    return pd.DataFrame(rows).sort_values(["scenario", "estimator_variant", "rmse", "mean_abs_log10_error"]).reset_index(drop=True)


policy_summary = summarize_policy_records(production_df)
policy_summary["rmse_rank"] = policy_summary.groupby(["scenario", "estimator_variant"])["rmse"].rank(method="min")
policy_summary["abs_log10_error_rank"] = policy_summary.groupby(["scenario", "estimator_variant"])["mean_abs_log10_error"].rank(method="min")
display(policy_summary)

leaderboard = (
    policy_summary
    .groupby(["estimator_variant", "rule_name"], as_index=False)
    .agg(
        mean_rmse_rank=("rmse_rank", "mean"),
        mean_abs_log10_error_rank=("abs_log10_error_rank", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_abs_log10_error=("mean_abs_log10_error", "mean"),
        mean_beta_selection_budget=("mean_beta_selection_budget", "mean"),
        mean_production_budget=("mean_production_budget", "mean"),
        mean_available_scan_n_weighted_samples=("mean_available_scan_n_weighted_samples", "mean"),
        mean_available_selected_scan_n_weighted_samples=("mean_available_selected_scan_n_weighted_samples", "mean"),
        mean_scan_n_weighted_samples=("mean_scan_n_weighted_samples", "mean"),
        mean_production_n_weighted_samples=("mean_production_n_weighted_samples", "mean"),
        mean_ess=("mean_ess", "mean"),
        mean_weight_cv=("mean_weight_cv", "mean"),
    )
    .sort_values(["estimator_variant", "mean_rmse_rank", "mean_abs_log10_error_rank", "mean_rmse"])
    .reset_index(drop=True)
)
display(leaderboard)

comparison_rows = []
for (scenario, rule_name), sub in policy_summary.groupby(["scenario", "rule_name"], sort=True):
    if "production_only" not in set(sub["estimator_variant"]):
        continue
    base = sub[sub["estimator_variant"] == "production_only"].iloc[0]
    for _, alt in sub[sub["estimator_variant"] != "production_only"].iterrows():
        comparison_rows.append({
            "scenario": scenario,
            "rule_name": rule_name,
            "exact_p": float(base["exact_p"]),
            "comparison_variant": str(alt["estimator_variant"]),
            "production_only_rmse": float(base["rmse"]),
            "comparison_rmse": float(alt["rmse"]),
            "rmse_ratio_vs_production_only": (
                float(alt["rmse"] / base["rmse"])
                if np.isfinite(base["rmse"]) and float(base["rmse"]) > 0.0
                else np.nan
            ),
            "production_only_abs_log10_error": float(base["mean_abs_log10_error"]),
            "comparison_abs_log10_error": float(alt["mean_abs_log10_error"]),
            "abs_log10_error_delta": float(alt["mean_abs_log10_error"] - base["mean_abs_log10_error"]),
            "scan_n_weighted_samples_added": float(alt["mean_scan_n_weighted_samples"]),
        })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["scenario", "comparison_variant", "rmse_ratio_vs_production_only"])
display(comparison_df)

## Save Outputs

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    run_dir.mkdir(parents=True, exist_ok=True)
    scan_df.to_json(run_dir / "scan_records.jsonl", orient="records", lines=True)
    production_df.to_json(run_dir / "production_records.jsonl", orient="records", lines=True)
    policy_summary.to_json(run_dir / "policy_summary.json", orient="records", indent=2)
    leaderboard.to_json(run_dir / "policy_leaderboard.json", orient="records", indent=2)
    comparison_df.to_json(run_dir / "estimator_comparison.json", orient="records", indent=2)
    (run_dir / "metadata.json").write_text(
        json.dumps(
            {
                "TOTAL_BUDGET": TOTAL_BUDGET,
                "N_REPEATS": N_REPEATS,
                "N_JOBS": N_JOBS,
                "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
                "SCAN_BUDGETS": SCAN_BUDGETS,
                "SCAN_FINALIST_COUNT": SCAN_FINALIST_COUNT,
                "SCAN_REFINE_TO_SCREEN_RATIO": SCAN_REFINE_TO_SCREEN_RATIO,
                "SCAN_FINAL_TO_SCREEN_RATIO": SCAN_FINAL_TO_SCREEN_RATIO,
                "Q_MULTIPLIERS": Q_MULTIPLIERS,
                "COARSE_Q_MULTIPLIERS": COARSE_Q_MULTIPLIERS,
                "BUDGET_RULE_SPECS": BUDGET_RULE_SPECS,
                "MCMC_LOCAL_SCAN_STRATEGY": MCMC_LOCAL_SCAN_STRATEGY,
                "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
                "PRODUCTION_ESTIMATE_VARIANCE": PRODUCTION_ESTIMATE_VARIANCE,
                "ESTIMATOR_VARIANTS": [
                    "production_only",
                    "selected_scan_plus_production",
                    "all_scan_plus_production",
                ],
                "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved fixed scan-budget grid outputs to {run_dir}")
else:
    print("SAVE_OUTPUTS is False; nothing saved.")